In [1]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse

seed = 42

p_ups = [1e-5, 1e-6, 1e-7, 1e-8, 1e-9]

max_cells_grid = [10, 20, 30, 50]

infile_template = "./h5ad_simu/diploid_12_generations_1cell_1e+08mutations_{p_up}.h5ad"
outdir_base = "./h5ad_simu_filtered"
os.makedirs(outdir_base, exist_ok=True)


def fmt_p_up(x: float) -> str:
    return f"{x:.0e}"


def subset_cells_by_generation(
    adata,
    max_cells_per_generation=50,
    seed=42,
    generation_key="generation",
):
    """
    Randomly subsample cells within each generation.

    For each generation, retain at most max_cells_per_generation cells.
    If the number of cells in a generation is smaller than or equal to the limit,
    all cells from that generation are retained.
    """
    if generation_key not in adata.obs.columns:
        raise KeyError(f"{generation_key} was not found in adata.obs.")

    rng = np.random.default_rng(seed)

    gen = np.asarray(adata.obs[generation_key]).astype(int)
    unique_gens = np.sort(np.unique(gen))

    keep_idx = []

    for g in unique_gens:
        idx_g = np.where(gen == g)[0]

        # Generate a fixed random order within each generation.
        # This makes smaller subsets nested within larger subsets when the same seed is used.
        idx_g_perm = rng.permutation(idx_g)

        take = min(max_cells_per_generation, idx_g.size)
        keep_idx.append(idx_g_perm[:take])

    keep_idx = np.sort(np.concatenate(keep_idx))

    adata_sub = adata[keep_idx, :].copy()
    adata_sub.obs_names_make_unique()

    return adata_sub


def filter_all_zero_snvs(adata):
    """
    Remove SNVs whose values are all zero across the retained cells.

    This function uses adata.X as the SNV matrix.
    """
    X = adata.X

    if sparse.issparse(X):
        snv_nnz = np.asarray((X != 0).sum(axis=0)).ravel()
    else:
        snv_nnz = np.count_nonzero(np.asarray(X), axis=0)

    keep_snv = snv_nnz > 0

    adata_filt = adata[:, keep_snv].copy()

    return adata_filt, snv_nnz


summary_records = []

for max_cells in max_cells_grid:
    outdir = os.path.join(
        outdir_base,
        f"max{max_cells}_cells_per_generation",
    )
    os.makedirs(outdir, exist_ok=True)

    print("\n" + "#" * 100)
    print(f"Processing max_cells_per_generation = {max_cells}")
    print("#" * 100)

    for mu in p_ups:
        p_up_str = fmt_p_up(mu)
        infile = infile_template.format(p_up=p_up_str)

        print("\n" + "=" * 80)
        print(f"Processing p_up = {p_up_str}")
        print(f"Input file: {infile}")

        adata = ad.read_h5ad(infile)
        adata.obs_names_make_unique()

        raw_n_obs, raw_n_vars = adata.shape
        print("Raw adata.shape:", adata.shape)

        adata_sub = subset_cells_by_generation(
            adata,
            max_cells_per_generation=max_cells,
            seed=seed,
            generation_key="generation",
        )

        subset_n_obs, subset_n_vars = adata_sub.shape
        print("Subset adata_sub.shape before SNV filtering:", adata_sub.shape)

        adata_sub, snv_nnz = filter_all_zero_snvs(adata_sub)

        filtered_n_obs, filtered_n_vars = adata_sub.shape
        print("Filtered adata_sub.shape after removing all-zero SNVs:", adata_sub.shape)

        adata_sub.obs["generation"] = adata_sub.obs["generation"].astype(int)

        adata_sub.uns["generation_subsampling"] = {
            "p_up": p_up_str,
            "seed": seed,
            "max_cells_per_generation": max_cells,
            "generation_key": "generation",
            "snv_filtering": "SNVs with all-zero values across retained cells were removed from adata.X.",
            "raw_n_obs": int(raw_n_obs),
            "raw_n_vars": int(raw_n_vars),
            "subset_n_obs_before_snv_filter": int(subset_n_obs),
            "subset_n_vars_before_snv_filter": int(subset_n_vars),
            "filtered_n_obs": int(filtered_n_obs),
            "filtered_n_vars": int(filtered_n_vars),
        }

        outfile = os.path.join(
            outdir,
            f"diploid_12_generations_1cell_1e+08mutations_{p_up_str}"
            f"_max{max_cells}cells_per_generation_nonzeroSNV.h5ad",
        )

        adata_sub.write_h5ad(outfile, compression="gzip")

        print(f"Saved filtered h5ad: {outfile}")

        gen_counts = adata_sub.obs["generation"].value_counts().sort_index()

        for g, n_cells in gen_counts.items():
            summary_records.append(
                {
                    "p_up": p_up_str,
                    "max_cells_per_generation": max_cells,
                    "generation": int(g),
                    "n_cells_after_subsampling": int(n_cells),
                    "raw_n_obs": int(raw_n_obs),
                    "raw_n_vars": int(raw_n_vars),
                    "subset_n_obs_before_snv_filter": int(subset_n_obs),
                    "subset_n_vars_before_snv_filter": int(subset_n_vars),
                    "filtered_n_obs": int(filtered_n_obs),
                    "filtered_n_vars": int(filtered_n_vars),
                    "outfile": outfile,
                }
            )

summary_df = pd.DataFrame(summary_records)

summary_out = os.path.join(
    outdir_base,
    "summary_generation_subsampling_nonzeroSNV_filtering.csv",
)

summary_df.to_csv(summary_out, index=False)

print("\nSaved summary table:")
print(summary_out)
print(summary_df.head())


####################################################################################################
Processing max_cells_per_generation = 10
####################################################################################################

Processing p_up = 1e-05
Input file: ./h5ad_simu/diploid_12_generations_1cell_1e+08mutations_1e-05.h5ad
Raw adata.shape: (8191, 100000000)
Subset adata_sub.shape before SNV filtering: (105, 100000000)
Filtered adata_sub.shape after removing all-zero SNVs: (105, 305596)
Saved filtered h5ad: ./h5ad_simu_filtered/max10_cells_per_generation/diploid_12_generations_1cell_1e+08mutations_1e-05_max10cells_per_generation_nonzeroSNV.h5ad

Processing p_up = 1e-06
Input file: ./h5ad_simu/diploid_12_generations_1cell_1e+08mutations_1e-06.h5ad
Raw adata.shape: (8191, 100000000)
Subset adata_sub.shape before SNV filtering: (105, 100000000)
Filtered adata_sub.shape after removing all-zero SNVs: (105, 30384)
Saved filtered h5ad: ./h5ad_simu_filtered/max10_cells_pe